# Phase 4 (SQL/Postgres Variant) — Feature Engineering

Builds the final quarterly modeling table per Phase 0's fixed feature set:

- Lagged ZHVI momentum (1, 3, and 6-month)
- MORTGAGE30US level
- MORTGAGE30US rate-of-change (12-month lag — the lag Phase 3 found strongest)
- The rate-of-change × prior-momentum interaction term

All computed with SQL window functions in `sql/005_phase4_features.sql`, across
the **full** dataset (Train, Embargo, and Test) — unlike Phase 3, which was
deliberately Train-only for EDA decisions. Building features for Test rows is
normal here; Phase 0's leakage rule is about not letting Test data inform
*decisions* (like the lag choice), not about withholding features from Test
rows.

Every feature attached to a quarter's label is measured as of the **preceding**
quarter — same leakage discipline as Phase 3 — which is what "predict one
quarter ahead" means at the row level.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
DB_NAME = os.environ.get("PGDATABASE", "capstone_sql_variant")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())

capstone_sql_variant


## 1. Build the Feature Tables

Runs every statement in `sql/005_phase4_features.sql` except the final
`SELECT` (read back separately below). Reuses the same comment-stripping
helper from Phase 3, since this file's final query is also preceded by
comment lines.

In [3]:
sql_text = open("../sql/005_phase4_features.sql").read()

def strip_leading_comments(s):
    lines = s.splitlines()
    while lines and lines[0].strip().startswith("--"):
        lines.pop(0)
    return "\n".join(lines).strip()

with engine.begin() as conn:
    for statement in sql_text.split(";"):
        statement = strip_leading_comments(statement.strip())
        if statement and not statement.upper().startswith("SELECT"):
            conn.execute(text(statement))

print("county_monthly_momentum, county_quarterly_momentum, national_monthly_rate_full,")
print("national_rate_roc_full, quarter_end_rate_features, quarterly_labeled_full,")
print("quarterly_modeling_table all (re)built.")

county_monthly_momentum, county_quarterly_momentum, national_monthly_rate_full,
national_rate_roc_full, quarter_end_rate_features, quarterly_labeled_full,
quarterly_modeling_table all (re)built.


## 2. Row Counts Per Split

Reads back the file's final query directly.

In [4]:
statements = [strip_leading_comments(s.strip()) for s in sql_text.split(";") if s.strip()]
statements = [s for s in statements if s]
final_query = [s for s in statements if s.upper().startswith("SELECT")][-1]

split_summary = pd.read_sql(final_query, engine)
split_summary

,split,n_rows,positive_rate_pct
0,embargo,3020,0.96
1,test,51987,8.11
2,train,165609,4.92
3,unassigned,6142,3.89


**Expect somewhat fewer rows than Phase 0's documented quarterly figures**
(Train 173,824 / Embargo 3,021 / Test 52,087) — this table additionally drops
rows lacking full 6-month momentum or 12-month rate history, which Phase 3's
simpler label table did not need to do.

## 3. Sanity Checks on the Modeling Table

In [5]:
modeling_table = pd.read_sql("SELECT * FROM quarterly_modeling_table;", engine)

print("Shape:", modeling_table.shape)
print("\nNull counts per column:")
print(modeling_table.isna().sum())
print("\nSplit counts:")
print(modeling_table["split"].value_counts())

assert modeling_table[["momentum_1m", "momentum_3m", "momentum_6m", "rate_level", "rate_roc_12m"]].isna().sum().sum() == 0, \
    "Some feature is still NULL after the history filter -- investigate."
print("\nNo NULLs in any feature column -- history filter worked correctly.")

Shape: (226758, 10)

Null counts per column:
county_fips                  0
quarter                      0
split                        0
decline_label                0
momentum_1m                  0
momentum_3m                  0
momentum_6m                  0
rate_level                   0
rate_roc_12m                 0
rate_momentum_interaction    0
dtype: int64

Split counts:
split
train         165609
test           51987
unassigned      6142
embargo         3020
Name: count, dtype: int64

No NULLs in any feature column -- history filter worked correctly.


In [6]:
modeling_table[["momentum_1m", "momentum_3m", "momentum_6m", "rate_level", "rate_roc_12m", "rate_momentum_interaction"]].describe()

,momentum_1m,momentum_3m,momentum_6m,rate_level,rate_roc_12m,rate_momentum_interaction
count,226758.000000,226758.000000,226758.000000,226758.000000,226758.000000,226758.000000
mean,0.003259,0.009981,0.020391,4.915734,0.029220,0.000008
std,0.007671,0.021054,0.038088,1.305676,0.249601,0.002334
min,-0.193830,-0.322727,-0.334358,2.684000,-0.278495,-0.048877
25%,-0.000689,-0.001220,-0.000174,3.864000,-0.125923,-0.000422
50%,0.003030,0.009342,0.019131,4.570000,-0.028359,-0.000002
75%,0.006952,0.020208,0.038577,6.180000,0.090403,0.000344
max,0.168129,0.236383,0.373781,7.200000,1.107586,0.059362


## 4. Leakage Spot-Check

Confirms every row's features genuinely come from the quarter *before* its
label quarter, not the same quarter — the core leakage rule this whole table
depends on.

In [7]:
check = pd.read_sql("""
    SELECT
        l.county_fips, l.quarter AS label_quarter, m.quarter AS momentum_quarter,
        r.quarter AS rate_quarter
    FROM quarterly_labeled_full l
    JOIN county_quarterly_momentum m
        ON m.county_fips = l.county_fips AND m.quarter + INTERVAL '3 months' = l.quarter
    JOIN quarter_end_rate_features r
        ON r.quarter + INTERVAL '3 months' = l.quarter
    LIMIT 5;
""", engine)
check

,county_fips,label_quarter,momentum_quarter,rate_quarter
0,01001,2000-04-01,2000-01-01,2000-01-01
1,01001,2000-07-01,2000-04-01,2000-04-01
2,01001,2000-10-01,2000-07-01,2000-07-01
3,01001,2001-01-01,2000-10-01,2000-10-01
4,01001,2001-04-01,2001-01-01,2001-01-01


**Expected:** `momentum_quarter` and `rate_quarter` should both be exactly one
quarter earlier than `label_quarter` on every row.

## Status

Phase 4 (SQL variant) complete once the null checks pass and the leakage
spot-check confirms every feature is one quarter behind its label. This table
(`quarterly_modeling_table`) is Phase 5's direct input — no further joining or
reshaping needed.